# Actividad 3 | Aplicar algoritmos de aprendizaje supervisado con PySpark

**Instituto Tecnológico y de Estudios Superiores de Monterrey**  
**Maestría en Inteligencia Artificial Aplicada**  
**Análisis de grandes volúmenes de datos**  
**Docente:** Dr. Iván Olmos Pineda

* ANGEL VEGA MARTINEZ — A01377304

**Fecha:** 24 de mayo de 2026



## 1. Introducción

El aprendizaje supervisado es una técnica de machine learning en la que el modelo aprende a partir de datos etiquetados, es decir, datos que ya incluyen la respuesta correcta. Su objetivo es encontrar la relación entre las variables de entrada y la variable de salida  para poder hacer predicciones sobre datos nuevos.

Las dos tareas principales del aprendizaje supervisado son:

**Clasificación**: cuando la salida es una categoría, por ejemplo sí/no o spam/no spam.

**Regresión**: cuando la salida es un valor numérico continuo, por ejemplo el precio de una casa.

Entre los algoritmos supervisados más representativos en la literatura se encuentran:

**a) Regresión lineal y regresión logística**
La regresión lineal se utiliza para problemas de regresión, donde se desea estimar una variable continua a partir de una o varias variables independientes. La regresión logística, aunque lleva el nombre “regresión”, se usa principalmente para clasificación, especialmente binaria y también multiclase en algunas implementaciones.

**b) Árboles de decisión**
Los árboles de decisión son modelos interpretables que dividen el espacio de datos mediante reglas sucesivas basadas en las características. Se utilizan tanto para clasificación como para regresión, y son especialmente útiles cuando se desea interpretar cómo se toman las decisiones del modelo.

**c) Random Forest**
Random Forest es un método de ensamble basado en múltiples árboles de decisión. Cada árbol se entrena con una muestra aleatoria de los datos y de las variables, y la predicción final se obtiene combinando las predicciones individuales. Este enfoque suele mejorar la capacidad de generalización y reducir la varianza respecto a un árbol único.

**d) Gradient Boosted Trees (GBT)**
Los Gradient-Boosted Trees construyen una secuencia de árboles donde cada nuevo árbol intenta corregir los errores del anterior. Son muy utilizados en tareas predictivas porque suelen ofrecer buen desempeño, aunque requieren una configuración más cuidadosa que otros métodos como Random Forest.

**e) Máquinas de soporte vectorial (SVM)**
Las Support Vector Machines buscan un hiperplano que separe de la mejor manera posible las clases, maximizando el margen entre ellas. Son muy conocidas en clasificación supervisada, especialmente en escenarios con espacios de alta dimensionalidad.

**f) Redes neuronales**
Las redes neuronales, incluyendo el perceptrón multicapa (Multilayer Perceptron), están formadas por capas de nodos interconectados y son capaces de modelar relaciones complejas no lineales entre variables. Son especialmente útiles cuando el problema presenta patrones complejos y no lineales.

**g) Naive Bayes y otros métodos probabilísticos**
Naive Bayes es un clasificador probabilístico basado en el teorema de Bayes y en el supuesto de independencia condicional entre variables. Aunque ese supuesto es fuerte, en muchos problemas prácticos —como clasificación de texto— ofrece resultados competitivos con bajo costo computacional.

**h) K-Nearest Neighbors (KNN)**
El método K-Nearest Neighbors clasifica o predice un nuevo ejemplo según la cercanía con ejemplos ya conocidos del conjunto de entrenamiento. Es un algoritmo representativo en la literatura del aprendizaje supervisado, aunque no suele ser el más adecuado para grandes volúmenes de datos distribuidos


En PySpark, estos algoritmos se implementan principalmente mediante la API spark.ml, que es la recomendada actualmente frente a la API antigua basada en RDDs.
Algunos algoritmos de aprendizaje supervisado disponibles en PySpark son:



*   RandomForestClassifier
*   GBTClassifier
*   MultilayerPerceptronClassifier
*   LogisticRegression y NaiveBayes
*   DecisionTreeClassifier

Para aplicar correctamente estos algoritmos en PySpark, normalmente se requiere preparar los datos, definir la columna objetivo, transformar las variables de entrada en una columna features, dividir los datos en entrenamiento y prueba, entrenar el modelo y evaluarlo con métricas adecuadas

## 1. Configuración del entorno

In [37]:
# Instalación de Java y Spark en Google Colab
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://dlcdn.apache.org/spark/spark-3.5.8/spark-3.5.8-bin-hadoop3.tgz
!tar -xzf spark-3.5.8-bin-hadoop3.tgz
!pip install -q findspark

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.8-bin-hadoop3"

import findspark
findspark.init()

from pyspark.sql import SparkSession

# Configuración optimizada para Colab/local mode
spark = SparkSession.builder     .master("local[*]")     .appName("PySpark_Optimizado_IncidentGrade")     .config("spark.sql.shuffle.partitions", "8")     .config("spark.default.parallelism", "8")     .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")     .config("spark.driver.memory", "8g")     .config("spark.sql.execution.arrow.pyspark.enabled", "true")     .getOrCreate()

spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
print(spark.version)

3.5.8


## 2. Parámetros del experimento

In [38]:
# ===============================
# Parámetros rápidos de ajuste
# ===============================
SAMPLE_FRAC = 0.02          # 1% de la base por clase
TOP_K_CATEGORY = 10         # 10 cardinalidad
SEED = 42

# Modelo principal: random_forest (más robusto)
MODEL_NAME = "random_forest"

# Hiperparámetros ligeros por defecto
DT_MAX_DEPTH = 8
RF_NUM_TREES = 10
RF_MAX_DEPTH = 6

# Mostrar predicciones y matriz de confusión
SHOW_PREDICTIONS = True
SHOW_CONFUSION = True

## 3. Carga del archivo CSV

Usar un esquema explícito reduce mucho el tiempo de lectura en comparación con `inferSchema=True`.

In [39]:
from google.colab import drive
drive.mount('/content/drive')

file_path = "/content/drive/MyDrive/Colab Notebooks/Análisis de grandes volúmenes de datos (Gpo 10)/Microsoft_GUIDE_Train.csv"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [40]:
from pyspark.sql.types import (
    StructType, StructField, LongType, IntegerType, DoubleType, StringType, TimestampType
)

schema = StructType([
    StructField("Id", LongType(), True),
    StructField("OrgId", IntegerType(), True),
    StructField("IncidentId", IntegerType(), True),
    StructField("AlertId", IntegerType(), True),
    StructField("Timestamp", TimestampType(), True),
    StructField("DetectorId", IntegerType(), True),
    StructField("AlertTitle", IntegerType(), True),
    StructField("Category", StringType(), True),
    StructField("MitreTechniques", StringType(), True),
    StructField("IncidentGrade", StringType(), True),
    StructField("ActionGrouped", StringType(), True),
    StructField("ActionGranular", StringType(), True),
    StructField("EntityType", StringType(), True),
    StructField("EvidenceRole", StringType(), True),
    StructField("DeviceId", IntegerType(), True),
    StructField("Sha256", IntegerType(), True),
    StructField("IpAddress", IntegerType(), True),
    StructField("Url", IntegerType(), True),
    StructField("AccountSid", IntegerType(), True),
    StructField("AccountUpn", IntegerType(), True),
    StructField("AccountObjectId", IntegerType(), True),
    StructField("AccountName", IntegerType(), True),
    StructField("DeviceName", IntegerType(), True),
    StructField("NetworkMessageId", IntegerType(), True),
    StructField("EmailClusterId", DoubleType(), True),
    StructField("RegistryKey", IntegerType(), True),
    StructField("RegistryValueName", IntegerType(), True),
    StructField("RegistryValueData", IntegerType(), True),
    StructField("ApplicationId", IntegerType(), True),
    StructField("ApplicationName", IntegerType(), True),
    StructField("OAuthApplicationId", IntegerType(), True),
    StructField("ThreatFamily", StringType(), True),
    StructField("FileName", IntegerType(), True),
    StructField("FolderPath", IntegerType(), True),
    StructField("ResourceIdName", IntegerType(), True),
    StructField("ResourceType", StringType(), True),
    StructField("Roles", StringType(), True),
    StructField("OSFamily", IntegerType(), True),
    StructField("OSVersion", IntegerType(), True),
    StructField("AntispamDirection", StringType(), True),
    StructField("SuspicionLevel", StringType(), True),
    StructField("LastVerdict", StringType(), True),
    StructField("CountryCode", IntegerType(), True),
    StructField("State", IntegerType(), True),
    StructField("City", IntegerType(), True),
    StructField("CategoryGroup", StringType(), True),
    StructField("EntityTypeGroup", StringType(), False)
])

df = spark.read     .option("header", True)     .schema(schema)     .csv(file_path)

print("Columnas cargadas:", len(df.columns))
df.printSchema()

Columnas cargadas: 47
root
 |-- Id: long (nullable = true)
 |-- OrgId: integer (nullable = true)
 |-- IncidentId: integer (nullable = true)
 |-- AlertId: integer (nullable = true)
 |-- Timestamp: timestamp (nullable = true)
 |-- DetectorId: integer (nullable = true)
 |-- AlertTitle: integer (nullable = true)
 |-- Category: string (nullable = true)
 |-- MitreTechniques: string (nullable = true)
 |-- IncidentGrade: string (nullable = true)
 |-- ActionGrouped: string (nullable = true)
 |-- ActionGranular: string (nullable = true)
 |-- EntityType: string (nullable = true)
 |-- EvidenceRole: string (nullable = true)
 |-- DeviceId: integer (nullable = true)
 |-- Sha256: integer (nullable = true)
 |-- IpAddress: integer (nullable = true)
 |-- Url: integer (nullable = true)
 |-- AccountSid: integer (nullable = true)
 |-- AccountUpn: integer (nullable = true)
 |-- AccountObjectId: integer (nullable = true)
 |-- AccountName: integer (nullable = true)
 |-- DeviceName: integer (nullable = true)
 |

## 4. Limpiar y reducir

Aquí se hacen las transformaciones que más impactan el tiempo de procesamiento:

- eliminar registros sin `IncidentGrade`
- conservar solo columnas compactas
- crear `CategoryReduced`
- crear una muestra M' pequeña y estratificada

In [41]:
from pyspark.sql.functions import col, when, hour, dayofweek, month

# Filtrar rápido la variable objetivo
base = df.filter(col("IncidentGrade").isNotNull())

# Reducir cardinalidad de Category antes del modelado
# Se calcula una sola vez sobre las filas válidas
from pyspark.sql.functions import desc

top_categories = [
    row["Category"]
    for row in base.groupBy("Category")
        .count()
        .orderBy(desc("count"))
        .limit(TOP_K_CATEGORY)
        .collect()
]

base = base.withColumn(
    "CategoryReduced",
    when(col("Category").isin(top_categories), col("Category")).otherwise("OtherCategory")
)

# Variables temporales compactas
base = base.withColumn("Hour", hour(col("Timestamp")))            .withColumn("DayOfWeek", dayofweek(col("Timestamp")))            .withColumn("Month", month(col("Timestamp")))

# Se excluyen columnas muy pesadas o de alta cardinalidad
selected_cols = [
    "IncidentId",
    "IncidentGrade",
    "OrgId",
    "DetectorId",
    "CategoryReduced",
    "ActionGrouped",
    "EntityType",
    "EvidenceRole",
    "ResourceType",
    "SuspicionLevel",
    "LastVerdict",
    "OSFamily",
    "OSVersion",
    "CountryCode",
    "CategoryGroup",
    "EntityTypeGroup",
    "Hour",
    "DayOfWeek",
    "Month"
]

base = base.select(*selected_cols).dropDuplicates()

# Muestreo estratificado por la variable objetivo
fractions = {
    row["IncidentGrade"]: SAMPLE_FRAC
    for row in base.select("IncidentGrade").distinct().collect()
}

df_muestra = base.sampleBy("IncidentGrade", fractions=fractions, seed=SEED)

# Cachear la muestra para reutilizarla
df_muestra = df_muestra.cache()
_ = df_muestra.count()

print("Muestra M' lista.")
print("Filas en M':", df_muestra.count())
df_muestra.groupBy("IncidentGrade").count().orderBy(desc("count")).show()

Muestra M' lista.
Filas en M': 79050
+--------------+-----+
| IncidentGrade|count|
+--------------+-----+
|BenignPositive|35688|
|  TruePositive|24010|
| FalsePositive|19352|
+--------------+-----+



## 5. Preparación del conjunto de entrenamiento y prueba

La división se hace por `IncidentId` para evitar que registros del mismo incidente aparezcan en train y test.

In [42]:
from pyspark.sql.functions import rand

incident_ids = df_muestra.select("IncidentId").distinct().withColumn("rand", rand(seed=SEED))

train_ids = incident_ids.filter(col("rand") < 0.8).select("IncidentId")
test_ids  = incident_ids.filter(col("rand") >= 0.8).select("IncidentId")

train_df = df_muestra.join(train_ids, on="IncidentId", how="inner")
test_df  = df_muestra.join(test_ids, on="IncidentId", how="inner")

# Cachear train/test para evitar recalcular varias veces
train_df = train_df.cache()
test_df = test_df.cache()
_ = train_df.count()
_ = test_df.count()

print("Train:", train_df.count())
print("Test:", test_df.count())

Train: 63207
Test: 15843


## 6. Definir variables y tratar nulos

Se usan menos columnas categóricas para acelerar entrenamiento.

In [43]:
categorical_cols = [
    "CategoryReduced",
    "ActionGrouped",
    "EntityType",
    "EvidenceRole",
    "ResourceType",
    "SuspicionLevel",
    "LastVerdict",
    "CategoryGroup",
    "EntityTypeGroup"
]

numeric_cols = [
    "OrgId",
    "DetectorId",
    "OSFamily",
    "OSVersion",
    "CountryCode",
    "Hour",
    "DayOfWeek",
    "Month"
]

# Relleno de nulos
train_df = train_df.fillna("Desconocido", subset=categorical_cols)
test_df = test_df.fillna("Desconocido", subset=categorical_cols)

for c in numeric_cols:
    train_df = train_df.fillna({c: 0})
    test_df = test_df.fillna({c: 0})

print("Nulos tratados.")

Nulos tratados.


## 7. Pipeline de PySpark

In [49]:
import time
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml import Pipeline

# Indexador de la etiqueta
label_indexer = StringIndexer(
    inputCol="IncidentGrade",
    outputCol="label",
    handleInvalid="keep"
)

# Indexadores para variables categóricas
feature_indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_cols
]

feature_cols = numeric_cols + [f"{c}_idx" for c in categorical_cols]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="keep"
)

if MODEL_NAME == "decision_tree":
    clf = DecisionTreeClassifier(
        labelCol="label",
        featuresCol="features",
        maxDepth=DT_MAX_DEPTH,
        seed=SEED
    )
elif MODEL_NAME == "random_forest":
    clf = RandomForestClassifier(
        labelCol="label",
        featuresCol="features",
        numTrees=10,
        maxDepth=8,
        maxBins=64,
        minInstancesPerNode=5,
        featureSubsetStrategy="sqrt",
        seed=42
    )
else:
    raise ValueError("MODEL_NAME debe ser 'decision_tree' o 'random_forest'.")

pipeline = Pipeline(stages=[label_indexer] + feature_indexers + [assembler, clf])

start = time.perf_counter()
model = pipeline.fit(train_df)
train_time_sec = time.perf_counter() - start

print(f"Modelo entrenado: {MODEL_NAME}")
print(f"Tiempo de entrenamiento: {train_time_sec/60:.2f} minutos")

Modelo entrenado: random_forest
Tiempo de entrenamiento: 0.36 minutos


## 8. Predicción y evaluación

In [50]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

predictions = model.transform(test_df).cache()
_ = predictions.count()

evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

accuracy = evaluator_acc.evaluate(predictions)
f1_score = evaluator_f1.evaluate(predictions)

print(f"Accuracy: {accuracy:.4f}")
print(f"F1-score: {f1_score:.4f}")

Accuracy: 0.6667
F1-score: 0.6421


## Interpretación de resultados

Se evaluó el desempeño del modelo de clasificación supervisada sobre el conjunto de prueba mediante las métricas de accuracy y F1-score. Con la configuración basada en `RandomForestClassifier`, se obtuvo un **accuracy de 0.6667** y un **F1-score de 0.6421**.

Estos resultados representan una mejora respecto a la configuración anterior, en la cual se había obtenido un accuracy de 0.6031 y un F1-score de 0.5527. En términos absolutos, el accuracy aumentó en **0.0636**, mientras que el F1-score mejoró en **0.0894**, lo cual indica un mejor desempeño general del modelo y una clasificación más equilibrada entre las distintas clases de la variable objetivo `IncidentGrade`.

El resultado obtenido sugiere que `RandomForestClassifier` logra capturar mejor los patrones presentes en los datos que el modelo base previamente evaluado. Esto es consistente con la naturaleza del algoritmo, ya que al combinar múltiples árboles de decisión reduce la variabilidad y suele mejorar la capacidad de generalización.

En conclusión, el modelo Random Forest puede considerarse la mejor opción dentro de los experimentos realizados, al mostrar un desempeño superior tanto en exactitud como en F1-score.

In [51]:
if SHOW_PREDICTIONS:
    predictions.select(
        "IncidentId", "IncidentGrade", "label", "prediction", "probability"
    ).show(10, truncate=False)

+----------+--------------+-----+----------+-----------------------------------------------------------------+
|IncidentId|IncidentGrade |label|prediction|probability                                                      |
+----------+--------------+-----+----------+-----------------------------------------------------------------+
|60991     |BenignPositive|0.0  |0.0       |[0.6226471570620281,0.11062397987475124,0.26672886306322063,0.0] |
|84912     |BenignPositive|0.0  |0.0       |[0.45752154839613074,0.29136839596954156,0.25111005563432764,0.0]|
|224513    |BenignPositive|0.0  |0.0       |[0.4426483455252484,0.328749882280122,0.2286017721946295,0.0]    |
|9465      |TruePositive  |1.0  |0.0       |[0.4646908953900382,0.33392559148185835,0.20138351312810351,0.0] |
|266576    |BenignPositive|0.0  |0.0       |[0.8703040270589716,0.11542214291421922,0.01427383002680926,0.0] |
|169       |TruePositive  |1.0  |1.0       |[0.020596347255164696,0.8883180818690768,0.09108557087575853,0.0]|
|

El análisis de algunas predicciones individuales permite observar que el modelo logra clasificar correctamente varios casos de la clase `BenignPositive` y algunos de `TruePositive` con niveles de confianza aceptables e incluso altos. Por ejemplo, en ciertos registros la probabilidad asignada a la clase correcta supera el 80%, lo cual muestra que el modelo identifica patrones claros en algunos incidentes.

Sin embargo, también se observan errores de clasificación en los que incidentes reales de tipo `TruePositive` o `FalsePositive` son predichos como `BenignPositive`. En varios de estos casos, las probabilidades estimadas por el modelo se encuentran relativamente cercanas entre clases, lo que sugiere que existen observaciones ambiguas o difíciles de separar con las variables utilizadas.

En general, estos ejemplos confirman que el modelo tiene una mejor capacidad para reconocer la clase mayoritaria, pero presenta más dificultad para distinguir correctamente algunos incidentes de clases minoritarias. Esto es consistente con la diferencia observada entre accuracy y F1-score, y sugiere oportunidades de mejora mediante ajuste de hiperparámetros, selección de variables o balanceo de clases.

In [52]:
if SHOW_CONFUSION:
    predictions.groupBy("label", "prediction")         .count()         .orderBy("label", "prediction")         .show(100)

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0| 6893|
|  0.0|       1.0|  168|
|  0.0|       2.0|  119|
|  1.0|       0.0| 2536|
|  1.0|       1.0| 1896|
|  1.0|       2.0|  282|
|  2.0|       0.0| 1941|
|  2.0|       1.0|  234|
|  2.0|       2.0| 1774|
+-----+----------+-----+



## Análisis de la matriz de confusión

La matriz de confusión permite observar con mayor detalle el comportamiento del modelo sobre cada una de las clases de la variable objetivo `IncidentGrade`.

En primer lugar, la clase `BenignPositive` presentó el mejor desempeño. De un total de 7,180 registros reales de esta clase, 6,893 fueron clasificados correctamente, lo que equivale a un recall aproximado de 96.0%. Esto indica que el modelo tiene una gran capacidad para reconocer incidentes benignos.

En contraste, las clases `TruePositive` y `FalsePositive` presentaron un desempeño más limitado. En el caso de `TruePositive`, de 4,714 casos reales únicamente 1,896 fueron clasificados correctamente, dando un recall aproximado de 40.2%. De manera similar, para `FalsePositive`, solo 1,774 de 3,949 registros fueron identificados correctamente, con un recall cercano a 44.9%.

Un patrón importante que se observa en la matriz es que una gran cantidad de registros reales de `TruePositive` y `FalsePositive` fueron clasificados como `BenignPositive`. Esto sugiere que el modelo tiende a favorecer la clase mayoritaria en aquellos casos donde la separación entre clases no es suficientemente clara.

No obstante, también se aprecia que cuando el modelo predice `TruePositive` o `FalsePositive`, suele hacerlo con una precisión alta. Esto indica que el modelo es relativamente conservador para asignar estas clases: las predice con menor frecuencia, pero con mayor confiabilidad.

En conjunto, la matriz de confusión confirma que el modelo Random Forest logra un desempeño general aceptable, pero todavía presenta dificultades para distinguir correctamente algunos incidentes de las clases minoritarias, especialmente cuando se parecen a casos benignos.

## 9. Importancia de variables

In [53]:
try:
    tree_model = model.stages[-1]
    importances = tree_model.featureImportances.toArray()
    feature_importance = sorted(zip(feature_cols, importances), key=lambda x: x[1], reverse=True)

    print("Top variables más importantes:")
    for feature, importance in feature_importance[:15]:
        print(f"{feature}: {importance:.6f}")
except Exception as e:
    print("No fue posible extraer las importancias:", e)

Top variables más importantes:
OrgId: 0.361301
DetectorId: 0.220984
CountryCode: 0.155181
CategoryReduced_idx: 0.113593
EntityType_idx: 0.070403
LastVerdict_idx: 0.035933
SuspicionLevel_idx: 0.019241
DayOfWeek: 0.006573
Hour: 0.006323
EvidenceRole_idx: 0.005312
Month: 0.002440
OSFamily: 0.002121
OSVersion: 0.000390
ActionGrouped_idx: 0.000207
ResourceType_idx: 0.000000


## Análisis de importancia de variables

El análisis de importancia de variables del modelo `RandomForestClassifier` permitió identificar cuáles atributos tuvieron mayor contribución en la predicción de la variable objetivo `IncidentGrade`.

La variable más importante fue `OrgId` (0.3613), seguida de `DetectorId` (0.2210) y `CountryCode` (0.1552). Esto sugiere que el modelo encuentra patrones relevantes asociados al contexto organizacional, al tipo de detector que generó la alerta y al país relacionado con el incidente.

Asimismo, variables como `CategoryReduced` (0.1136) y `EntityType` (0.0704) también aportaron información importante al modelo, lo que indica que la categoría del evento y el tipo de entidad involucrada ayudan a discriminar entre las distintas clases de `IncidentGrade`.

Por otro lado, variables como `DayOfWeek`, `Hour`, `EvidenceRole`, `Month`, `OSFamily` y `OSVersion` presentaron una importancia considerablemente menor, mientras que `ResourceType` tuvo una importancia nula dentro del modelo. Esto sugiere que dichas variables tienen un aporte limitado o redundante en el contexto del problema analizado.

En conjunto, estos resultados muestran que el modelo basa gran parte de sus decisiones en variables estructurales y contextuales, más que en variables temporales o de configuración del sistema.

## 10. Conclusiones

En esta actividad se aplicó un modelo de aprendizaje supervisado en PySpark para predecir la variable `IncidentGrade`. Se realizaron las etapas de limpieza, reducción de cardinalidad, muestreo estratificado, partición en entrenamiento y prueba, transformación de variables y entrenamiento del modelo.

El algoritmo que ofreció el mejor desempeño fue `RandomForestClassifier`, alcanzando un **accuracy de 0.6667** y un **F1-score de 0.6421**. Estos resultados muestran que el modelo logró una capacidad predictiva aceptable, especialmente para la clase `BenignPositive`, aunque todavía presenta dificultades para distinguir correctamente algunos casos de `TruePositive` y `FalsePositive`.

En general, se concluye que PySpark permite desarrollar experimentos de aprendizaje supervisado de forma escalable y que, con una adecuada preparación de datos, es posible obtener modelos funcionales incluso a partir de bases de datos grandes.